# Career Path Prediction — Grouped Target Training & Model Comparison
This notebook evaluates the grouped career target (4 categories) using multiple models (Random Forest, XGBoost, LightGBM) to compare their performance against the original 12-class target.

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
from pathlib import Path
from imblearn.over_sampling import SMOTENC

In [21]:
DATA_PATH = "data/PS2_Dataset.csv"
TARGET = "Suggested Job Role"
RANDOM_STATE = 42
TEST_SIZE = 0.2

ROLE_TO_CATEGORY = {
    "Applications Developer":                     "Software Development",
    "CRM Technical Developer":                    "Software Development",
    "Database Developer":                         "Software Development",
    "Mobile Applications Developer":              "Software Development",
    "Software Developer":                         "Software Development",
    "Software Engineer":                          "Software Development",
    "Web Developer":                              "Software Development",
    "Network Security Engineer":                  "Cybersecurity",
    "Systems Security Administrator":             "Cybersecurity",
    "Software Quality Assurance (QA) / Testing":  "QA & Support",
    "Technical Support":                          "QA & Support",
    "UX Designer":                                "Design",
}

In [22]:
class CareerModelWrapper:
    """Wrapper that handles preprocessing, prediction, and label decoding seamlessly."""
    def __init__(self, preprocessor, model, label_encoder):
        self.preprocessor = preprocessor
        self.model = model
        self.label_encoder = label_encoder
        
    def predict(self, X):
        X_proc = self.preprocessor.transform(X)
        preds = self.model.predict(X_proc)
        return self.label_encoder.inverse_transform(preds)

def print_header(title: str):
    print(f"\n{'='*75}")
    print(f"  {title}")
    print(f"{'='*75}")

def print_class_distribution(y: pd.Series, title: str, encoder=None):
    print(f"\n{title}")
    print("-" * 75)
    class_dist = y.value_counts()
    total = len(y)
    for cls_val, count in class_dist.items():
        cls_name = encoder.inverse_transform([cls_val])[0] if encoder else cls_val
        pct = count / total * 100
        bar = "█" * int(pct)
        print(f"  {str(cls_name)[:45]:45s} {count:5d} ({pct:5.1f}%)  {bar}")
    print(f"\n  Total samples: {total} | Classes: {y.nunique()}")

def build_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols)
    ])

def train_and_evaluate(X_train_proc, X_test_proc, y_train, y_test, num_cols, cat_cols, model, model_label):
    print_header(f"TRAINING: {model_label}")
    
    print("  [1/3] Applying SMOTENC oversampling...")
    cat_feature_indices = list(range(len(num_cols), len(num_cols) + len(cat_cols)))
    smote = SMOTENC(categorical_features=cat_feature_indices, random_state=RANDOM_STATE, k_neighbors=3)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_proc, y_train)
    
    print(f"  [2/3] Training {model.__class__.__name__}...")
    model.fit(X_train_bal, y_train_bal)
    
    print(f"  [3/3] Evaluating on test set...")
    y_pred = model.predict(X_test_proc)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    
    print(f"\n  Accuracy:  {acc:.4f} | F1-Score: {f1:.4f}")
    
    metrics = {
        "Model": model_label,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    }
    return metrics, model

### 1. Load Dataset & Encode Target

In [23]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=[TARGET])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

y_orig_raw = df[TARGET]
y_grouped_raw = y_orig_raw.map(ROLE_TO_CATEGORY)

# Encode targets to integers (Required for XGBoost & LightGBM)
le_orig = LabelEncoder()
y_orig = le_orig.fit_transform(y_orig_raw)

le_grouped = LabelEncoder()
y_grouped = le_grouped.fit_transform(y_grouped_raw)

print_class_distribution(pd.Series(y_orig), "ORIGINAL Target Distribution (12 classes)", le_orig)
print_class_distribution(pd.Series(y_grouped), "GROUPED Target Distribution (4 categories)", le_grouped)


ORIGINAL Target Distribution (12 classes)
---------------------------------------------------------------------------
  Network Security Engineer                       630 (  9.1%)  █████████
  Software Engineer                               590 (  8.5%)  ████████
  UX Designer                                     589 (  8.5%)  ████████
  Software Developer                              587 (  8.5%)  ████████
  Database Developer                              581 (  8.4%)  ████████
  Software Quality Assurance (QA) / Testing       571 (  8.3%)  ████████
  Web Developer                                   570 (  8.3%)  ████████
  CRM Technical Developer                         567 (  8.2%)  ████████
  Technical Support                               565 (  8.2%)  ████████
  Systems Security Administrator                  562 (  8.1%)  ████████
  Applications Developer                          551 (  8.0%)  ███████
  Mobile Applications Developer                   538 (  7.8%)  ███████

  Tot

C:\Users\karam\AppData\Local\Temp\ipykernel_36004\3124521140.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


### 2. Preprocessing & Splitting

In [24]:
X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(X, y_orig, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_orig)
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(X, y_grouped, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_grouped)

# Preprocess once to save time in loops
preprocessor_o = build_preprocessor(num_cols, cat_cols)
X_train_o_proc = preprocessor_o.fit_transform(X_train_o)
X_test_o_proc = preprocessor_o.transform(X_test_o)

preprocessor_g = build_preprocessor(num_cols, cat_cols)
X_train_g_proc = preprocessor_g.fit_transform(X_train_g)
X_test_g_proc = preprocessor_g.transform(X_test_g)

all_metrics = []
models_saved = {}

### 3. Evaluate Models

In [25]:
# 1. Original Target (Baseline) - Random Forest
rf_orig = RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
m1, _ = train_and_evaluate(X_train_o_proc, X_test_o_proc, y_train_o, y_test_o, num_cols, cat_cols, rf_orig, "ORIGINAL Target (12 classes) - Random Forest")
all_metrics.append(m1)


  TRAINING: ORIGINAL Target (12 classes) - Random Forest
  [1/3] Applying SMOTENC oversampling...
  [2/3] Training RandomForestClassifier...
  [3/3] Evaluating on test set...

  Accuracy:  0.0840 | F1-Score: 0.0838


In [26]:
# 2. Grouped Target - Random Forest
rf_grouped = RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
m2, trained_rf = train_and_evaluate(X_train_g_proc, X_test_g_proc, y_train_g, y_test_g, num_cols, cat_cols, rf_grouped, "GROUPED Target (4 categories) - Random Forest")
all_metrics.append(m2)
models_saved['Random Forest'] = trained_rf


  TRAINING: GROUPED Target (4 categories) - Random Forest
  [1/3] Applying SMOTENC oversampling...
  [2/3] Training RandomForestClassifier...
  [3/3] Evaluating on test set...

  Accuracy:  0.4395 | F1-Score: 0.3987


In [27]:
# 3. Grouped Target - XGBoost
xgb_grouped = XGBClassifier(n_estimators=300, max_depth=10, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, use_label_encoder=False, eval_metric="mlogloss")
m3, trained_xgb = train_and_evaluate(X_train_g_proc, X_test_g_proc, y_train_g, y_test_g, num_cols, cat_cols, xgb_grouped, "GROUPED Target (4 categories) - XGBoost")
all_metrics.append(m3)
models_saved['XGBoost'] = trained_xgb


  TRAINING: GROUPED Target (4 categories) - XGBoost
  [1/3] Applying SMOTENC oversampling...
  [2/3] Training XGBClassifier...


d:\CareerPrediction\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [03:40:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  [3/3] Evaluating on test set...

  Accuracy:  0.5431 | F1-Score: 0.4289


In [28]:
# 4. Grouped Target - LightGBM
lgbm_grouped = LGBMClassifier(n_estimators=300, max_depth=15, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
m4, trained_lgbm = train_and_evaluate(X_train_g_proc, X_test_g_proc, y_train_g, y_test_g, num_cols, cat_cols, lgbm_grouped, "GROUPED Target (4 categories) - LightGBM")
all_metrics.append(m4)
models_saved['LightGBM'] = trained_lgbm


  TRAINING: GROUPED Target (4 categories) - LightGBM
  [1/3] Applying SMOTENC oversampling...
  [2/3] Training LGBMClassifier...
  [3/3] Evaluating on test set...

  Accuracy:  0.5395 | F1-Score: 0.4146


d:\CareerPrediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### 4. Final Comparison & Save Best Model

In [29]:
print_header("FINAL MODEL COMPARISON")
comp_df = pd.DataFrame(all_metrics)
comp_df['Accuracy'] = comp_df['Accuracy'].apply(lambda x: f"{x:.4f}")
comp_df['Precision'] = comp_df['Precision'].apply(lambda x: f"{x:.4f}")
comp_df['Recall'] = comp_df['Recall'].apply(lambda x: f"{x:.4f}")
comp_df['F1-Score'] = comp_df['F1-Score'].apply(lambda x: f"{x:.4f}")
print(comp_df.to_string(index=False))

print_header("SAVING BEST GROUPED MODEL")
grouped_metrics = pd.DataFrame(all_metrics)
grouped_metrics = grouped_metrics[grouped_metrics["Model"].str.contains("GROUPED")].copy()
grouped_metrics["Accuracy"] = grouped_metrics["Accuracy"].astype(float)
best_model_name = grouped_metrics.sort_values("Accuracy", ascending=False).iloc[0]["Model"].split(" - ")[-1]

print(f"  Best Grouped Model: {best_model_name}")
best_model = models_saved[best_model_name]

Path("models").mkdir(exist_ok=True)

final_pipeline = CareerModelWrapper(
    preprocessor=preprocessor_g,
    model=best_model,
    label_encoder=le_grouped
)

joblib.dump(final_pipeline, "models/career_model_grouped.pkl")
joblib.dump(ROLE_TO_CATEGORY, "models/role_to_category_map.pkl")
print("  \u2713 Saved best grouped model and mapping to models/ directory.")


  FINAL MODEL COMPARISON
                                        Model Accuracy Precision Recall F1-Score
 ORIGINAL Target (12 classes) - Random Forest   0.0840    0.0839 0.0840   0.0838
GROUPED Target (4 categories) - Random Forest   0.4395    0.3713 0.4395   0.3987
      GROUPED Target (4 categories) - XGBoost   0.5431    0.3885 0.5431   0.4289
     GROUPED Target (4 categories) - LightGBM   0.5395    0.3483 0.5395   0.4146

  SAVING BEST GROUPED MODEL
  Best Grouped Model: XGBoost
  ✓ Saved best grouped model and mapping to models/ directory.
